-------------------------------------------------------------------------------

CHIBO 2024

An attempt to static calculate the forces in (x,y,z) at all the suspension pickup points for various roll/jounce effects

18.12.2024: Created

-------------------------------------------------------------------------------

In [13]:
import numpy as np
from scipy.optimize import fsolve
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [8]:
# Declare motions for evaluation

# Define maximum roll and jounce inputs
body_roll = 10  # Maximum roll angle in degrees
body_jounce = 0.05  # Maximum jounce in meters (5 cm)

# Generate roll and jounce motion inputs
roll_angles = np.linspace(0, np.radians(body_roll), 100)  # 0 to max roll angle in 100 steps
jounce_displacements = np.linspace(0, body_jounce, 100)  # 0 to max jounce in 100 steps

# Check roll_angles and jounce_displacements
print("Roll Angles (rad):", roll_angles[:5], "...", roll_angles[-5:])
print("Jounce Displacements (m):", jounce_displacements[:5], "...", jounce_displacements[-5:])


Roll Angles (rad): [0.         0.00176296 0.00352592 0.00528888 0.00705184] ... [0.16748109 0.16924405 0.17100701 0.17276997 0.17453293]
Jounce Displacements (m): [0.         0.00050505 0.0010101  0.00151515 0.0020202 ] ... [0.0479798  0.04848485 0.0489899  0.04949495 0.05      ]


In [15]:
# Suspension pickup points
#                          x      y    z
upper_a_arm = np.array([
    [0.0015, 0.5065, 0.315],  # Outer point (converted from mm to m)
    [-0.135, 0.240, 0.220],   # Inner leading point
    [0.180, 0.255, 0.255]     # Inner trailing point
])

lower_a_arm = np.array([
    [-0.0175, 0.519, 0.137],  # Outer point
    [-0.180, 0.165, 0.087],   # Inner leading point
    [0.180, 0.170, 0.087]     # Inner trailing point
])

tie_rod = np.array([
    [0.060, 0.515, 0.160],  # Outer point
    [0.070, 0.195, 0.100]   # Inner point
])

tire_cop = np.array([-0.005, 0.525, 0])  # Converted from mm to m
force_vector = np.array([50, 500, -2000.0])  # Force at the CoP (N)


In [19]:
def plot_suspension_interactive(upper, lower, tie, cop):
    fig = go.Figure()

    # Upper A-arm
    fig.add_trace(go.Scatter3d(
        x=[upper[0, 0], upper[1, 0]], y=[upper[0, 1], upper[1, 1]], z=[upper[0, 2], upper[1, 2]],
        mode='lines+markers',
        name='Upper A-arm (Outer to Inner Leading)',
        marker=dict(size=6, color='blue'),
        line=dict(color='blue', width=2)
    ))
    fig.add_trace(go.Scatter3d(
        x=[upper[0, 0], upper[2, 0]], y=[upper[0, 1], upper[2, 1]], z=[upper[0, 2], upper[2, 2]],
        mode='lines',
        name='Upper A-arm (Outer to Inner Trailing)',
        line=dict(color='blue', width=2, dash='dot')
    ))

    # Lower A-arm
    fig.add_trace(go.Scatter3d(
        x=[lower[0, 0], lower[1, 0]], y=[lower[0, 1], lower[1, 1]], z=[lower[0, 2], lower[1, 2]],
        mode='lines+markers',
        name='Lower A-arm (Outer to Inner Leading)',
        marker=dict(size=6, color='green'),
        line=dict(color='green', width=2)
    ))
    fig.add_trace(go.Scatter3d(
        x=[lower[0, 0], lower[2, 0]], y=[lower[0, 1], lower[2, 1]], z=[lower[0, 2], lower[2, 2]],
        mode='lines',
        name='Lower A-arm (Outer to Inner Trailing)',
        line=dict(color='green', width=2, dash='dot')
    ))

    # Tie rod
    fig.add_trace(go.Scatter3d(
        x=[tie[0, 0], tie[1, 0]], y=[tie[0, 1], tie[1, 1]], z=[tie[0, 2], tie[1, 2]],
        mode='lines+markers',
        name='Tie Rod',
        marker=dict(size=6, color='red'),
        line=dict(color='red', width=2)
    ))

    # Tire CoP
    fig.add_trace(go.Scatter3d(
        x=[cop[0]], y=[cop[1]], z=[cop[2]],
        mode='markers',
        name='Tire CoP',
        marker=dict(size=8, color='orange')
    ))

    # Set axis labels
    fig.update_layout(
        title="Interactive Suspension Geometry (Corrected Connections)",
        scene=dict(
            xaxis_title="X (m)",
            yaxis_title="Y (m)",
            zaxis_title="Z (m)"
        )
    )

    fig.show()

# Example usage
plot_suspension_interactive(upper_a_arm, lower_a_arm, tie_rod, tire_cop)


In [21]:
import numpy as np

def calculate_roll_center(upper, lower, cop):
    """
    Calculate the roll center (RC) based on 3D suspension geometry and tire CoP.
    Assumes suspension symmetry about the vehicle centerline.

    Parameters:
        upper (np.ndarray): 3x3 matrix of upper control arm pickup points.
                           [[outer], [inner front], [inner rear]] points.
        lower (np.ndarray): 3x3 matrix of lower control arm pickup points.
                           [[outer], [inner front], [inner rear]] points.
        cop (np.ndarray): 1x3 array for tire center of pressure (CoP).

    Returns:
        np.ndarray: Roll center coordinates [x, y, z] in the global coordinate system.
        dict: Additional geometry data including instant centers and angles.
    """
    # Input validation
    if not all(isinstance(x, np.ndarray) for x in [upper, lower, cop]):
        raise ValueError("Inputs must be numpy arrays")
    if upper.shape != (3, 3) or lower.shape != (3, 3):
        raise ValueError("Control arm matrices must be 3x3")
    if cop.shape != (3,):
        raise ValueError("CoP must be 1x3 array")

    # Calculate control arm planes normal vectors
    def get_plane_normal(points):
        """Calculate normal vector of plane defined by three points."""
        v1 = points[1] - points[0]
        v2 = points[2] - points[0]
        return np.cross(v1, v2)

    upper_normal = get_plane_normal(upper)
    lower_normal = get_plane_normal(lower)

    # Project arms onto front view (XZ plane)
    upper_outer_2d = upper[0, [0, 2]]  # [x, z]
    upper_inner_2d = upper[1, [0, 2]]  # Using front inner point
    lower_outer_2d = lower[0, [0, 2]]
    lower_inner_2d = lower[1, [0, 2]]
    tire_cop_2d = cop[[0, 2]]

    # Calculate control arm angles in front view
    def get_arm_angle(outer, inner):
        """Calculate arm angle from horizontal in front view."""
        delta = inner - outer
        return np.degrees(np.arctan2(delta[1], delta[0]))

    upper_angle = get_arm_angle(upper_outer_2d, upper_inner_2d)
    lower_angle = get_arm_angle(lower_outer_2d, lower_inner_2d)

    # Find instant center (IC) in front view
    def line_equation(p1, p2):
        """Return slope and intercept of line through two points."""
        m = (p2[1] - p1[1]) / (p2[0] - p1[0])
        c = p1[1] - m * p1[0]
        return m, c

    m_upper, c_upper = line_equation(upper_outer_2d, upper_inner_2d)
    m_lower, c_lower = line_equation(lower_outer_2d, lower_inner_2d)
    
    ic_x = (c_lower - c_upper) / (m_upper - m_lower)
    ic_z = m_upper * ic_x + c_upper
    instant_center_2d = np.array([ic_x, ic_z])

    # Calculate roll center height
    # RC is where line from IC to CoP intersects vehicle centerline (x=0)
    m_ic_to_cop = (tire_cop_2d[1] - ic_z) / (tire_cop_2d[0] - ic_x)
    c_ic_to_cop = ic_z - m_ic_to_cop * ic_x
    rc_z = c_ic_to_cop  # At x=0

    # Consider 3D effects
    # Calculate swing arm length (SAL) considering y-axis position
    sal = np.sqrt(ic_x**2 + (upper[1, 1] - upper[0, 1])**2)  # Using y-distance
    
    # Calculate roll center including y-coordinate
    # Using average y-position of inner pickup points
    rc_y = (np.mean(upper[:, 1]) + np.mean(lower[:, 1])) / 2
    
    roll_center = np.array([0, rc_y, rc_z])

    # Package additional geometry data
    geometry_data = {
        'instant_center_2d': instant_center_2d,
        'swing_arm_length': sal,
        'upper_arm_angle': upper_angle,
        'lower_arm_angle': lower_angle,
        'upper_normal': upper_normal,
        'lower_normal': lower_normal
    }

    return roll_center, geometry_data

# Example usage:
def example_usage():
    # Example suspension geometry (in mm)
    upper_arm = np.array([
        [400, 0, 500],    # Outer point
        [200, 300, 520],  # Inner front
        [200, 250, 520]   # Inner rear
    ])
    
    lower_arm = np.array([
        [420, 0, 200],    # Outer point
        [180, 300, 230],  # Inner front
        [180, 250, 230]   # Inner rear
    ])
    
    tire_cop = np.array([400, 0, 0])  # Center of pressure
    
    rc, geometry = calculate_roll_center(upper_arm, lower_arm, tire_cop)
    print("Roll Center [x, y, z]:", rc)
    print("\nGeometry Data:")
    for key, value in geometry.items():
        print(f"{key}:", value)

if __name__ == "__main__":
    example_usage()

Roll Center [x, y, z]: [  0.         183.33333333  56.80672269]

Geometry Data:
instant_center_2d: [-11500.   1690.]
swing_arm_length: 11503.912377969507
upper_arm_angle: 174.28940686250036
lower_arm_angle: 172.8749836510982
upper_normal: [ 1000     0 10000]
lower_normal: [ 1500     0 12000]


In [ ]:
def apply_roll_about_axis(points, angle, axis):
    """
    Apply roll (rotation around the x-axis passing through the roll center).
    
    Parameters:
        points (np.array): Nx3 matrix of 3D points.
        angle (float): Roll angle in radians.
        axis (np.array): Roll axis (roll center coordinates).

    Returns:
        np.array: Transformed points.
    """
    # Translate points to make roll center the origin
    translated_points = points - axis
    
    # Apply rotation about x-axis
    roll_matrix = np.array([[1, 0, 0],
                            [0, np.cos(angle), -np.sin(angle)],
                            [0, np.sin(angle), np.cos(angle)]])
    rotated_points = translated_points @ roll_matrix.T
    
    # Translate points back
    return rotated_points + axis

def apply_jounce(points, displacement):
    """Apply vertical jounce to a set of points."""
    jounce_vector = np.array([0, 0, displacement])
    return points + jounce_vector


In [ ]:
def resolve_forces(force):
    """Split force equally between upper and lower A-arm inner points (placeholder)."""
    return force / 2

In [ ]:
# Storage for transformed pickup points and forces
upper_inner_forces = []
lower_inner_forces = []

# Iterate through roll and jounce motions
for roll_angle, jounce_disp in zip(roll_angles, jounce_displacements):
    # Apply roll and jounce transformations
    upper_transformed = apply_jounce(apply_roll_about_axis(upper_a_arm, roll_angle, roll_center), jounce_disp)
    lower_transformed = apply_jounce(apply_roll_about_axis(lower_a_arm, roll_angle, roll_center), jounce_disp)
    
    # Resolve forces at the inner pickup points (split evenly as placeholder)
    upper_force = resolve_forces(force_vector)
    lower_force = resolve_forces(force_vector)
    
    # Store resolved forces
    upper_inner_forces.append(upper_force)
    lower_inner_forces.append(lower_force)

In [ ]:
# Extract z-force (vertical) components for plotting
upper_z_forces = [force[2] for force in upper_inner_forces]
lower_z_forces = [force[2] for force in lower_inner_forces]

# Plot forces vs. motion
plt.figure(figsize=(10, 6))
plt.plot(jounce_displacements, upper_z_forces, label="Upper A-Arm Inner Force (Z)")
plt.plot(jounce_displacements, lower_z_forces, label="Lower A-Arm Inner Force (Z)")
plt.xlabel("Jounce Displacement (m)")
plt.ylabel("Force (N)")
plt.title("Resultant Vertical Forces at Inner Suspension Points")
plt.legend()
plt.grid()
plt.show()